## Reinicialização do ambiente

> **Atenção:** a próxima célula exclui permanentemente o catálogo `youtube_lakehouse` e todos os seus schemas, tabelas e dados. Execute-a somente quando quiser reiniciar o ambiente de desenvolvimento ou testes do zero.

In [0]:
%sql
DROP CATALOG IF EXISTS youtube_lakehouse CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS youtube_lakehouse
COMMENT 'Lakehouse educacional para ingestão e análise de dados do YouTube';


CREATE SCHEMA IF NOT EXISTS youtube_lakehouse.control
COMMENT 'Configuração, estado e controle operacional das ingestões';


CREATE SCHEMA IF NOT EXISTS youtube_lakehouse.raw
COMMENT 'Payloads brutos e imutáveis da YouTube Data API';


CREATE SCHEMA IF NOT EXISTS youtube_lakehouse.silver
COMMENT 'Dados normalizados e deduplicados';


CREATE SCHEMA IF NOT EXISTS youtube_lakehouse.gold
COMMENT 'Modelos analíticos e métricas de negócio';

## Tabelas do schema control\n
\n
Configuração dos vídeos monitorados, estado de processamento e auditoria das execuções.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS youtube_lakehouse.control.video_targets (
  video_id STRING NOT NULL,
  is_active BOOLEAN NOT NULL,
  priority INT NOT NULL,
  refresh_interval_hours INT NOT NULL,
  created_at TIMESTAMP NOT NULL,
  updated_at TIMESTAMP NOT NULL
) USING DELTA
COMMENT 'Lista inicial e configurável de vídeos a serem processados'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.control.video_targets.video_id IS 'PK: Identificador público do vídeo do YouTube a ser processado';
COMMENT ON COLUMN youtube_lakehouse.control.video_targets.is_active IS 'Indica se o vídeo participa das próximas execuções';
COMMENT ON COLUMN youtube_lakehouse.control.video_targets.priority IS 'Prioridade decrescente de seleção dentro da fila';
COMMENT ON COLUMN youtube_lakehouse.control.video_targets.refresh_interval_hours IS 'Intervalo, em horas, entre coletas bem-sucedidas do vídeo';
COMMENT ON COLUMN youtube_lakehouse.control.video_targets.created_at IS 'Data e hora UTC de cadastro do vídeo na lista inicial';
COMMENT ON COLUMN youtube_lakehouse.control.video_targets.updated_at IS 'Data e hora UTC da última alteração de configuração do vídeo';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.control.channel_targets (
  channel_id STRING NOT NULL,
  discovery_mode STRING NOT NULL DEFAULT 'NONE',
  created_at TIMESTAMP NOT NULL,
  updated_at TIMESTAMP NOT NULL
) USING DELTA
COMMENT 'Configuração manual para descoberta automática de novos vídeos por canal'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.feature.allowColumnDefaults' = 'supported'
);

COMMENT ON COLUMN youtube_lakehouse.control.channel_targets.channel_id IS 'PK, FK → silver.channels.channel_id. Canal elegível para configuração de descoberta';
COMMENT ON COLUMN youtube_lakehouse.control.channel_targets.discovery_mode IS 'Modo de descoberta: NONE não consulta uploads, ALL cadastra todos após o corte e LAST cadastra somente o upload mais recente';
COMMENT ON COLUMN youtube_lakehouse.control.channel_targets.created_at IS 'Data e hora UTC de criação da configuração do canal';
COMMENT ON COLUMN youtube_lakehouse.control.channel_targets.updated_at IS 'Data e hora UTC da última alteração da configuração do canal';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.control.channel_discovery_runs (
  discovery_id STRING NOT NULL,
  started_at TIMESTAMP NOT NULL,
  ended_at TIMESTAMP,
  status STRING NOT NULL,
  channels_attempted BIGINT NOT NULL,
  channels_succeeded BIGINT NOT NULL,
  channels_failed BIGINT NOT NULL,
  videos_discovered BIGINT NOT NULL,
  videos_registered BIGINT NOT NULL,
  api_cost_units BIGINT NOT NULL,
  error_message STRING,
  CONSTRAINT channel_discovery_runs_pk PRIMARY KEY (discovery_id) NOT ENFORCED RELY
) USING DELTA
COMMENT 'Auditoria das execuções do Job de descoberta de novos vídeos por canal'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.discovery_id IS 'PK: Identificador da execução de descoberta';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.started_at IS 'Data e hora UTC de início da descoberta';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.ended_at IS 'Data e hora UTC de encerramento da descoberta';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.status IS 'Estado da descoberta: RUNNING, SUCCESS, PARTIAL_SUCCESS ou FAILED';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.channels_attempted IS 'Quantidade de canais consultados na API';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.channels_succeeded IS 'Quantidade de canais consultados sem erro';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.channels_failed IS 'Quantidade de canais cuja descoberta falhou';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.videos_discovered IS 'Quantidade de IDs de vídeos posteriores ao corte por canal';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.videos_registered IS 'Quantidade de IDs inseridos em control.video_targets';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.api_cost_units IS 'Custo estimado da YouTube Data API, acumulado por chamada HTTP da descoberta';
COMMENT ON COLUMN youtube_lakehouse.control.channel_discovery_runs.error_message IS 'Resumo dos erros por canal, quando houver';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.control.ingestion_runs (
  ingestion_id STRING NOT NULL,
  channel_handle STRING NOT NULL,
  channel_name STRING,
  started_at TIMESTAMP NOT NULL,
  ended_at TIMESTAMP,
  status STRING NOT NULL,
  error_message STRING,
  CONSTRAINT ingestion_runs_pk PRIMARY KEY (ingestion_id) NOT ENFORCED RELY
) USING DELTA
COMMENT 'Registro operacional auditável de cada execução de ingestão da plataforma'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

CREATE TABLE IF NOT EXISTS youtube_lakehouse.control.video_processing_state (
  video_id STRING NOT NULL,
  status STRING NOT NULL,
  first_processed_at TIMESTAMP,
  claimed_at TIMESTAMP,
  last_attempt_at TIMESTAMP,
  last_succeeded_at TIMESTAMP,
  next_refresh_at TIMESTAMP,
  attempt_count BIGINT NOT NULL,
  last_ingestion_id STRING,
  error_message STRING
) USING DELTA
COMMENT 'Estado operacional por vídeo: fila, tentativa, sucesso, erro e próxima atualização'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.video_id IS 'PK, FK → video_targets.video_id. Identificador do vídeo monitorado';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.status IS 'Resultado ou estado atual: PROCESSING, SUCCESS, FAILED ou NOT_FOUND';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.first_processed_at IS 'Data e hora UTC da primeira conclusão bem-sucedida do vídeo';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.claimed_at IS 'Data e hora UTC em que o vídeo foi reservado pela execução em processamento; nula fora desse estado';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.last_attempt_at IS 'Data e hora UTC da tentativa mais recente de processamento';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.last_succeeded_at IS 'Data e hora UTC da coleta bem-sucedida mais recente';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.next_refresh_at IS 'Momento a partir do qual o vídeo volta a ficar elegível para coleta';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.attempt_count IS 'Número acumulado de tentativas de processamento';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.last_ingestion_id IS 'FK → ingestion_runs.ingestion_id. Identificador da execução que possui ou processou o vídeo mais recentemente';
COMMENT ON COLUMN youtube_lakehouse.control.video_processing_state.error_message IS 'Último erro técnico associado ao vídeo, quando houver';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.control.ingestion_step_outcomes (
  ingestion_id STRING NOT NULL,
  video_id STRING NOT NULL,
  step STRING NOT NULL,
  status STRING NOT NULL,
  completed_at TIMESTAMP NOT NULL,
  error_message STRING
) USING DELTA
COMMENT 'Resultado por vídeo e etapa do Workflow; permite retomar e finalizar uma ingestão distribuída'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.control.ingestion_step_outcomes.ingestion_id IS 'PK (composta com video_id e step), FK → ingestion_runs.ingestion_id. Execução à qual pertence o resultado da etapa';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_step_outcomes.video_id IS 'PK (composta com ingestion_id e step), FK → video_targets.video_id. Vídeo processado pela etapa';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_step_outcomes.step IS 'PK (composta com ingestion_id e video_id). Etapa do Workflow: fetch_videos, fetch_channels, fetch_comments ou fetch_replies';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_step_outcomes.status IS 'Resultado da etapa para o vídeo: SUCCESS, FAILED ou NOT_FOUND';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_step_outcomes.completed_at IS 'Data e hora UTC do último resultado registrado para a etapa';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_step_outcomes.error_message IS 'Erro técnico da etapa, quando houver';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.control.ingestion_comments (
  ingestion_id STRING NOT NULL,
  video_id STRING NOT NULL,
  comment_id STRING NOT NULL
) USING DELTA
COMMENT 'Handoff temporário dos comentários retornados em cada execução para o fetch de replies'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.control.ingestion_comments.ingestion_id IS 'PK (composta com video_id e comment_id), FK → ingestion_runs.ingestion_id. Execução que retornou o comentário';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_comments.video_id IS 'PK (composta com ingestion_id e comment_id), FK → videos.video_id. Vídeo que contém o comentário';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_comments.comment_id IS 'PK (composta com ingestion_id e video_id), FK → comments.comment_id. Comentário de primeiro nível a ser usado no fetch de replies';

COMMENT ON COLUMN youtube_lakehouse.control.ingestion_runs.ingestion_id IS 'PK: Identificador único e imutável da execução de ingestão';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_runs.channel_handle IS 'Origem ou configuração que disparou a execução; preservada por compatibilidade';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_runs.channel_name IS 'Nome público do canal observado na API; se a execução processar vários canais, contém os nomes separados por vírgula';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_runs.started_at IS 'Data e hora UTC de início da execução';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_runs.ended_at IS 'Data e hora UTC de encerramento; nula enquanto a execução está em andamento';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_runs.status IS 'Estado operacional da execução: RUNNING, SUCCESS, PARTIAL_SUCCESS ou FAILED';
COMMENT ON COLUMN youtube_lakehouse.control.ingestion_runs.error_message IS 'Mensagem técnica de erro para auditoria; nula quando a execução é bem-sucedida';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.control.task_execution_logs (
  ingestion_id STRING,
  task_key STRING NOT NULL,
  task_run_id STRING,
  started_at TIMESTAMP NOT NULL,
  ended_at TIMESTAMP NOT NULL,
  status STRING NOT NULL,
  videos_attempted BIGINT NOT NULL,
  videos_succeeded BIGINT NOT NULL,
  videos_failed BIGINT NOT NULL,
  records_fetched BIGINT NOT NULL,
  api_cost_units BIGINT NOT NULL,
  error_message STRING
) USING DELTA
COMMENT 'Resumo idempotente de cada tentativa de task do Workflow, para observabilidade operacional'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.ingestion_id IS 'Execução de ingestão associada; pode ser nula quando claim_targets falha antes de criar a execução';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.task_key IS 'Chave estável da task do Databricks Job';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.task_run_id IS 'Identificador da tentativa da task no Databricks Job; distingue retries';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.started_at IS 'Data e hora UTC de início da tentativa da task';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.ended_at IS 'Data e hora UTC de fim da tentativa da task';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.status IS 'Resultado semântico da task: SUCCESS, PARTIAL_SUCCESS ou FAILED';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.videos_attempted IS 'Quantidade de vídeos considerados pela task';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.videos_succeeded IS 'Quantidade de vídeos concluídos sem erro na task';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.videos_failed IS 'Quantidade de vídeos com erro ou estado não concluído na task';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.records_fetched IS 'Quantidade de registros de domínio retornados pela task';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.api_cost_units IS 'Custo estimado da YouTube Data API, acumulado por chamada HTTP da task';
COMMENT ON COLUMN youtube_lakehouse.control.task_execution_logs.error_message IS 'Erro técnico da task quando a tentativa falha';



## Tabelas do schema raw\n
\n
Respostas originais e imutáveis da YouTube Data API para auditoria e reprocessamento.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS youtube_lakehouse.raw.api_responses (
  ingestion_id STRING NOT NULL,
  resource STRING NOT NULL,
  request_params_json STRING NOT NULL,
  response_json STRING NOT NULL,
  received_at TIMESTAMP NOT NULL
) USING DELTA
COMMENT 'Registro bruto e imutável das respostas da YouTube Data API para auditoria e reprocessamento'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.raw.api_responses.ingestion_id IS 'Identificador da execução de ingestão ou descoberta que originou a chamada à API';
COMMENT ON COLUMN youtube_lakehouse.raw.api_responses.resource IS 'Recurso da YouTube Data API consultado';
COMMENT ON COLUMN youtube_lakehouse.raw.api_responses.request_params_json IS 'Parâmetros da chamada serializados em JSON para auditoria e reprocessamento';
COMMENT ON COLUMN youtube_lakehouse.raw.api_responses.response_json IS 'Resposta original da API em JSON; preservar sem transformação';
COMMENT ON COLUMN youtube_lakehouse.raw.api_responses.received_at IS 'Data e hora UTC de recebimento da resposta da API';

## Tabelas do schema silver\n
\n
Entidades normalizadas atuais e snapshots temporais de vídeos e canais.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS youtube_lakehouse.silver.channels (
  channel_id STRING NOT NULL,
  title STRING,
  description STRING,
  custom_url STRING,
  published_at STRING,
  country STRING,
  view_count BIGINT,
  subscriber_count BIGINT,
  video_count BIGINT,
  uploads_playlist_id STRING,
  ingested_at TIMESTAMP NOT NULL,
  CONSTRAINT channels_pk PRIMARY KEY (channel_id) NOT ENFORCED
) USING DELTA
COMMENT 'Dimensão normalizada de canais do YouTube, atualizada por channel_id'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.silver.channels.channel_id IS 'PK: Identificador único do canal fornecido pela YouTube Data API';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.title IS 'Título público do canal';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.description IS 'Descrição pública do canal';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.custom_url IS 'URL personalizada pública do canal, quando disponível';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.published_at IS 'Data de criação do canal no formato retornado pela API';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.country IS 'País informado publicamente pelo canal, quando disponível';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.view_count IS 'Total público de visualizações do canal no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.subscriber_count IS 'Total público de inscritos do canal no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.video_count IS 'Total público de vídeos do canal no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.uploads_playlist_id IS 'Identificador da playlist de uploads do canal';
COMMENT ON COLUMN youtube_lakehouse.silver.channels.ingested_at IS 'Data e hora UTC em que o registro foi carregado na camada silver';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.silver.videos (
  video_id STRING NOT NULL,
  channel_id STRING NOT NULL,
  title STRING,
  description STRING,
  published_at TIMESTAMP,
  category_id INT,
  duration INTERVAL DAY TO SECOND,
  definition STRING,
  caption STRING,
  view_count BIGINT,
  like_count BIGINT,
  comment_count BIGINT,
  privacy_status STRING,
  ingested_at TIMESTAMP NOT NULL,
  CONSTRAINT videos_pk PRIMARY KEY (video_id) NOT ENFORCED,
  CONSTRAINT videos_channel_fk FOREIGN KEY (channel_id) REFERENCES youtube_lakehouse.silver.channels(channel_id) NOT ENFORCED
) USING DELTA
COMMENT 'Entidade normalizada de vídeos do YouTube, atualizada por video_id'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.silver.videos.video_id IS 'PK: Identificador único do vídeo fornecido pela YouTube Data API';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.channel_id IS 'FK → channels.channel_id. Identificador do canal proprietário do vídeo';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.title IS 'Título público do vídeo';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.description IS 'Descrição pública do vídeo';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.published_at IS 'Data e hora UTC de publicação do vídeo';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.category_id IS 'Identificador numérico da categoria do vídeo no YouTube';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.duration IS 'Duração do vídeo como intervalo de dia a segundo';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.definition IS 'Definição de qualidade do vídeo, por exemplo hd ou sd';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.caption IS 'Indicador de disponibilidade de legendas retornado pela API';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.view_count IS 'Total público de visualizações no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.like_count IS 'Total público de curtidas no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.comment_count IS 'Total público de comentários no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.privacy_status IS 'Status de privacidade do vídeo retornado pela API';
COMMENT ON COLUMN youtube_lakehouse.silver.videos.ingested_at IS 'Data e hora UTC em que o registro foi carregado na camada silver';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.silver.video_tags (
  video_id STRING NOT NULL,
  tag STRING NOT NULL,
  ingested_at TIMESTAMP NOT NULL,
  CONSTRAINT video_tags_pk PRIMARY KEY (video_id, tag) NOT ENFORCED,
  CONSTRAINT video_tags_video_fk FOREIGN KEY (video_id) REFERENCES youtube_lakehouse.silver.videos(video_id) NOT ENFORCED
) USING DELTA
COMMENT 'Bridge normalizada das tags públicas vigentes por vídeo'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.silver.video_tags.video_id IS 'PK (composta com tag), FK → videos.video_id. Vídeo associado à tag';
COMMENT ON COLUMN youtube_lakehouse.silver.video_tags.tag IS 'PK (composta com video_id). Tag pública do vídeo, em uma linha por associação';
COMMENT ON COLUMN youtube_lakehouse.silver.video_tags.ingested_at IS 'Data e hora UTC da última observação da associação';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.silver.comments (
  thread_id STRING,
  comment_id STRING NOT NULL,
  parent_id STRING,
  video_id STRING,
  author_name STRING,
  author_channel_id STRING,
  text STRING,
  like_count BIGINT,
  published_at STRING,
  updated_at STRING,
  reply_count BIGINT,
  ingested_at TIMESTAMP NOT NULL
) USING DELTA
COMMENT 'Comentários públicos normalizados do YouTube, atualizados por comment_id'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.silver.comments.thread_id IS 'Identificador da thread de comentários do vídeo';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.comment_id IS 'PK: Identificador único do comentário fornecido pela YouTube Data API';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.parent_id IS 'FK → comments.comment_id (recursivo). Identificador do comentário pai; nulo para comentários de primeiro nível';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.video_id IS 'FK → videos.video_id. Identificador do vídeo ao qual o comentário pertence';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.author_name IS 'Nome público exibido do autor do comentário';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.author_channel_id IS 'Identificador público do canal do autor, quando disponível';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.text IS 'Texto público do comentário; tratar como conteúdo externo não confiável';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.like_count IS 'Total público de curtidas do comentário no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.published_at IS 'Data de publicação do comentário no formato retornado pela API';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.updated_at IS 'Data da última atualização do comentário no formato retornado pela API';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.reply_count IS 'Quantidade de respostas de primeiro nível no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.comments.ingested_at IS 'Data e hora UTC em que o registro foi carregado na camada silver';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.silver.replies (
  comment_id STRING NOT NULL,
  parent_id STRING,
  video_id STRING,
  author_name STRING,
  author_channel_id STRING,
  text STRING,
  like_count BIGINT,
  published_at STRING,
  updated_at STRING,
  ingested_at TIMESTAMP NOT NULL
) USING DELTA
COMMENT 'Respostas públicas a comentários do YouTube, atualizadas por comment_id'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.silver.replies.comment_id IS 'PK: Identificador único da resposta fornecido pela YouTube Data API';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.parent_id IS 'FK → comments.comment_id. Identificador do comentário de primeiro nível ao qual a resposta pertence';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.video_id IS 'FK → videos.video_id. Identificador do vídeo ao qual a resposta pertence';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.author_name IS 'Nome público exibido do autor da resposta';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.author_channel_id IS 'Identificador público do canal do autor, quando disponível';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.text IS 'Texto público da resposta; tratar como conteúdo externo não confiável';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.like_count IS 'Total público de curtidas da resposta no momento da coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.published_at IS 'Data de publicação da resposta no formato retornado pela API';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.updated_at IS 'Data da última atualização da resposta no formato retornado pela API';
COMMENT ON COLUMN youtube_lakehouse.silver.replies.ingested_at IS 'Data e hora UTC em que o registro foi carregado na camada silver';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.silver.channel_snapshots (
  channel_id STRING NOT NULL,
  ingestion_id STRING NOT NULL,
  collected_at TIMESTAMP NOT NULL,
  collected_date DATE NOT NULL,
  view_count BIGINT,
  subscriber_count BIGINT,
  video_count BIGINT
) USING DELTA
COMMENT 'Snapshots imutáveis das métricas de canal; silver.channels contém somente o estado atual'
PARTITIONED BY (collected_date)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.silver.channel_snapshots.collected_at IS 'Data e hora UTC em que as métricas do canal foram observadas';
COMMENT ON COLUMN youtube_lakehouse.silver.channel_snapshots.collected_date IS 'Data UTC da coleta usada para particionamento';
COMMENT ON COLUMN youtube_lakehouse.silver.channel_snapshots.channel_id IS 'PK (lógica, composta com ingestion_id), FK → channels.channel_id. Canal ao qual pertence a observação histórica';
COMMENT ON COLUMN youtube_lakehouse.silver.channel_snapshots.ingestion_id IS 'PK lógica (composta com channel_id). Execução de ingestão que registrou a observação';
COMMENT ON COLUMN youtube_lakehouse.silver.channel_snapshots.view_count IS 'Total público de visualizações do canal observado na coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.channel_snapshots.subscriber_count IS 'Quantidade pública de inscritos observada na coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.channel_snapshots.video_count IS 'Quantidade pública de vídeos do canal observada na coleta';

CREATE TABLE IF NOT EXISTS youtube_lakehouse.silver.video_snapshots (
  video_id STRING NOT NULL,
  ingestion_id STRING NOT NULL,
  collected_at TIMESTAMP NOT NULL,
  collected_date DATE NOT NULL,
  view_count BIGINT,
  like_count BIGINT,
  comment_count BIGINT
) USING DELTA
COMMENT 'Snapshots imutáveis das métricas de vídeo; silver.videos contém somente o estado atual'
PARTITIONED BY (collected_date)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COMMENT ON COLUMN youtube_lakehouse.silver.video_snapshots.collected_at IS 'Data e hora UTC em que as métricas do vídeo foram observadas';
COMMENT ON COLUMN youtube_lakehouse.silver.video_snapshots.collected_date IS 'Data UTC da coleta usada para particionamento';
COMMENT ON COLUMN youtube_lakehouse.silver.video_snapshots.video_id IS 'PK (lógica, composta com ingestion_id), FK → videos.video_id. Vídeo ao qual pertence a observação histórica';
COMMENT ON COLUMN youtube_lakehouse.silver.video_snapshots.ingestion_id IS 'PK lógica (composta com video_id). Execução de ingestão que registrou a observação';
COMMENT ON COLUMN youtube_lakehouse.silver.video_snapshots.view_count IS 'Visualizações públicas observadas na coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.video_snapshots.like_count IS 'Curtidas públicas observadas na coleta';
COMMENT ON COLUMN youtube_lakehouse.silver.video_snapshots.comment_count IS 'Quantidade pública de comentários observada na coleta';

CREATE OR REPLACE VIEW youtube_lakehouse.silver.vw_dashboard_ingestion_runs
COMMENT 'Camada de apresentação do dashboard: execuções e duração operacional'
AS
SELECT
  ingestion_id,
  channel_name,
  started_at,
  ended_at,
  status,
  error_message,
  CASE
    WHEN ended_at IS NOT NULL THEN unix_timestamp(ended_at) - unix_timestamp(started_at)
  END AS duration_seconds
FROM youtube_lakehouse.control.ingestion_runs;

COMMENT ON COLUMN youtube_lakehouse.silver.vw_dashboard_ingestion_runs.duration_seconds IS 'Duração da execução em segundos, calculada como diferença entre ended_at e started_at';

CREATE OR REPLACE VIEW youtube_lakehouse.silver.vw_dashboard_video_targets
COMMENT 'Camada de apresentação do dashboard: fila e estado operacional por vídeo'
AS
SELECT
  target.video_id,
  target.is_active,
  target.priority,
  target.refresh_interval_hours,
  state.status,
  state.last_succeeded_at,
  state.next_refresh_at,
  state.attempt_count,
  state.error_message
FROM youtube_lakehouse.control.video_targets AS target
LEFT JOIN youtube_lakehouse.control.video_processing_state AS state
  ON target.video_id = state.video_id;

CREATE OR REPLACE VIEW youtube_lakehouse.silver.vw_dashboard_video_metrics
COMMENT 'Camada de apresentação do dashboard: snapshots temporais de métricas de vídeo'
AS
SELECT
  snapshot.video_id,
  video.title AS video_title,
  video.channel_id,
  channel.title AS channel_title,
  snapshot.collected_at,
  snapshot.view_count,
  snapshot.like_count,
  snapshot.comment_count
FROM youtube_lakehouse.silver.video_snapshots AS snapshot
LEFT JOIN youtube_lakehouse.silver.videos AS video
  ON snapshot.video_id = video.video_id
LEFT JOIN youtube_lakehouse.silver.channels AS channel
  ON video.channel_id = channel.channel_id;

CREATE OR REPLACE VIEW youtube_lakehouse.silver.vw_dashboard_channel_metrics
COMMENT 'Camada de apresentação do dashboard: snapshots temporais de métricas de canal'
AS
SELECT
  snapshot.channel_id,
  channel.title AS channel_title,
  snapshot.collected_at,
  snapshot.view_count,
  snapshot.subscriber_count,
  snapshot.video_count
FROM youtube_lakehouse.silver.channel_snapshots AS snapshot
LEFT JOIN youtube_lakehouse.silver.channels AS channel
  ON snapshot.channel_id = channel.channel_id;

